# Exhaustive noisy-channel Bayesian transcripts

This notebook builds two deterministic question schedules and exhaustively enumerates all eight $K=3$ report histories for each. No latent value or channel coin is sampled or stored. The final cells configure (but do not automatically launch) batched Qwen3.5 execution and tensor capture.

In [ ]:
from __future__ import annotations

import tempfile
from pathlib import Path

from mats_experiments.noisy_channel_bayesian import (
    CandidateEvidenceBayesianEnvironment,
    CandidateEvidenceDatasetGenerator,
    CandidateEvidenceQuestion,
    CaptureSpec,
    ExecutionConfig,
    MetricSpec,
    ModelConfig,
    NoisyChannelBayesianEnvironment,
    QwenRunner,
    RandomSubsetQuestion,
    SystemPrompt,
    TokenizerBinding,
    TranscriptDataset,
    TranscriptDatasetGenerator,
    XVsYPosteriorProbe,
    exact_pattern_mass,
    summarize_representation_control,
)


class DeterministicDemoTokenizer:
    """Small offline tokenizer used only to make the builder reproducible."""

    chat_template = "deterministic-demo-v1"
    name_or_path = "deterministic-demo"

    def apply_chat_template(self, messages, *, tokenize, add_generation_prompt, **_):
        text = "".join(f"<{m['role']}>{m['content']}" for m in messages)
        if add_generation_prompt:
            text += "<assistant>"
        return list(text.encode()) if tokenize else text

In [ ]:
environment = NoisyChannelBayesianEnvironment(n=8, k=3, r_values="3/4")
probe = XVsYPosteriorProbe(
    x=2, y=7, reasoning_budget=40, allow_same=False, call_layout="conversation"
)
demo_tokenizer_binding = TokenizerBinding(DeterministicDemoTokenizer())
demo_system_prompt = SystemPrompt("Reason carefully from the stated noisy-channel model.")
builder = TranscriptDatasetGenerator(
    environment=environment,
    question=RandomSubsetQuestion(subset_size=4, replacement=False, sort=True),
    probe=probe,
    tokenizer_binding=demo_tokenizer_binding,
    system_prompt=demo_system_prompt,
    seed=20260902,
)
dataset = builder.generate(num_question_sets=2)
assert len(dataset) == 2 * 2**3
dataset[["row_id", "question_set_index", "answer_pattern", "prior_predictive_exact"]][:10]

## Exact targets and mass normalization

The exact fields are rational strings. Floating-point mirrors are included for analysis and plotting.

In [ ]:
first = dataset[0]
{
    "pattern": first["answer_pattern"],
    "evidence_mass": first["prior_predictive_exact"],
    "posterior": first["posterior_exact"],
    "x_vs_y": (first["x_posterior_exact"], first["y_posterior_exact"]),
    "target": first["ground_truth_choice"],
}

In [ ]:
masses = {index: exact_pattern_mass(dataset, index) for index in range(2)}
assert all(mass == 1 for mass in masses.values())
masses

## Uniform-history versus natural-distribution summaries

For illustration, the next cell attaches deterministic mock correctness values. Real result datasets get these fields from `QwenRunner`.

In [ ]:
illustrative_rows = []
for row in dataset:
    illustrative_rows.append(
        {
            **row,
            "posterior_correct": (row["answer_pattern"] in {"YYY", "NNN"})
            if row["ground_truth_choice"] is not None
            else None,
        }
    )
TranscriptDataset(illustrative_rows).summarize()

## Save/load and table inspection

In [ ]:
demo_temporary_directory = tempfile.TemporaryDirectory(prefix="noisy_channel_bayesian_demo_")
experiment_dir = Path(demo_temporary_directory.name)
dataset.save(experiment_dir)
reloaded = TranscriptDataset.load(experiment_dir)
assert reloaded[0] == dataset[0]
reloaded.columns, reloaded.head(2), reloaded["answer_pattern"][:8]

## Paired candidate-evidence control

The reduced environment derives per-candidate `AGREES`/`DISAGREES` evidence from each raw row. The exact posterior and natural-distribution weight remain identical. Raw sets and reports remain in `audit_metadata`, but the reduced renderer has no interface through which they can enter the prompt.

In [ ]:
immediate_probe = XVsYPosteriorProbe(x=2, y=7, reasoning_budget=0, allow_same=False)
raw_immediate = TranscriptDatasetGenerator(
    environment=environment,
    question=RandomSubsetQuestion(subset_size=4, replacement=False, sort=True),
    probe=immediate_probe,
    tokenizer_binding=demo_tokenizer_binding,
    system_prompt=demo_system_prompt,
    seed=20260902,
).generate(num_question_sets=2)
reduced_question = CandidateEvidenceQuestion()
reduced = CandidateEvidenceDatasetGenerator(
    environment=CandidateEvidenceBayesianEnvironment(n=8, k=3, r_values="3/4"),
    question=reduced_question,
    probe=immediate_probe,
    tokenizer_binding=demo_tokenizer_binding,
    system_prompt=demo_system_prompt,
).generate(source_dataset=raw_immediate)
assert [row["source_row_id"] for row in reduced] == [row["row_id"] for row in raw_immediate]
assert all(
    raw["posterior_exact"] == compact["posterior_exact"]
    and raw["prior_predictive_exact"] == compact["prior_predictive_exact"]
    for raw, compact in zip(raw_immediate, reduced)
)
print(reduced[0]["messages"][-1]["content"])

In [ ]:
reduced_metrics = MetricSpec(
    x_surface="X", y_surface="Y", same_surface="SAME", sequence_scores=True
)
reduced_capture = CaptureSpec(
    logits_boundaries=("answer",),
    streams=("resid_pre", "token_mixer_out", "mlp_out", "resid_post"),
    layers="all",
    tokens="all",
    every_decode_position=False,
)
raw_mock = TranscriptDataset(
    [
        {
            **row,
            "posterior_correct": row["answer_pattern_index"] % 3 != 0
            if row["ground_truth_choice"] is not None
            else None,
            "parse_compliance": True,
        }
        for row in raw_immediate
    ]
)
reduced_mock = TranscriptDataset(
    [
        {
            **row,
            "posterior_correct": True if row["ground_truth_choice"] is not None else None,
            "parse_compliance": True,
        }
        for row in reduced
    ]
)
summarize_representation_control(raw_mock, reduced_mock)

## Batched Qwen3.5 execution (explicit opt-in)

Replace `model_name_or_path` with a local or Hub checkpoint. The same runner supports dense 4B/9B checkpoints and the official 27B GPTQ-Int4 checkpoint through Transformers-compatible loading. Stage one is batched across all reasoning prompts before stage-two answer prompts are reconstructed and batched.

In [ ]:
model_config = ModelConfig(model_name_or_path="Qwen/Qwen3.5-4B", dtype="auto")
execution = ExecutionConfig(
    experiment_dir=experiment_dir,
    run_id="qwen35_4b_batch2",
    batch_size=2,
    capture=CaptureSpec(
        logits_boundaries=("reasoning", "answer"),
        streams=("resid_pre", "token_mixer_out", "mlp_out", "resid_post"),
        layers=(0, -1),
        tokens="last",
    ),
)
RUN_MODEL = False
if RUN_MODEL:
    results = dataset.execute(QwenRunner(model_config), execution)
    print(results.summarize())

## Optional every-decode-position capture

This disk-heavy configuration is deliberately not executed here. It performs a batched teacher-forced pass after generation and stores tensors only at generated positions.

In [ ]:
every_decode_execution = ExecutionConfig(
    experiment_dir=experiment_dir,
    run_id="qwen35_every_decode_example",
    batch_size=2,
    capture=CaptureSpec(
        logits_boundaries=("answer",),
        streams=("resid_post",),
        layers=(-1,),
        every_decode_position=True,
    ),
)
every_decode_execution